In [17]:
# ============================================================
# 0) Google Drive 마운트
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ============================================================
# 1) 필수 패키지 설치
# ============================================================
!pip -q install "open_clip_torch>=2.23.0" ftfy einops tqdm pillow

# ============================================================
# 2) CONFIG – 경로·모델·배치크기
# ============================================================
CFG = {
    # 입력 경로
    "CSV_TEST":   "/content/drive/MyDrive/INHA_AI_Challenge/open/test.csv",
    "DD_EMB_CSV": "/content/drive/MyDrive/INHA_AI_Challenge/open/DDColor/embed_submission.csv",
    "FX_EMB_CSV": "/content/drive/MyDrive/INHA_AI_Challenge/open/flux_image/embed_submission.csv",
    "DIR_DD_IMG": "/content/drive/MyDrive/INHA_AI_Challenge/open/DDColor",
    "DIR_FX_IMG": "/content/drive/MyDrive/INHA_AI_Challenge/open/flux_image",

    # 출력 경로
    "DIR_OUT_IMG": "/content/drive/MyDrive/INHA_AI_Challenge/open/best_image",
    "CSV_OUT":     "/content/drive/MyDrive/INHA_AI_Challenge/open/embed_best.csv",

    # CLIP
    "CLIP_MODEL":    "ViT-L-14",
    "CLIP_PRETRAIN": "openai",

    # 기타
    "EXT": ".png",       # 이미지 확장자
    "BATCH": 64,         # 캡션 배치
}

# ============================================================
# 3) 라이브러리 로드 & CLIP 모델
# ============================================================
import os, torch, pandas as pd, numpy as np, shutil
from tqdm.auto import tqdm
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"
print("💻 device:", device)

clip_model, _, _ = open_clip.create_model_and_transforms(
    CFG["CLIP_MODEL"], pretrained=CFG["CLIP_PRETRAIN"]
)
clip_model = clip_model.to(device).eval()
tokenizer   = open_clip.get_tokenizer(CFG["CLIP_MODEL"])

@torch.no_grad()
def encode_text(texts):
    tok = tokenizer(texts).to(device)
    emb = clip_model.encode_text(tok)
    return emb / emb.norm(dim=-1, keepdim=True)

# ============================================================
# 4) CSV 로드
# ============================================================
def load_embed_csv(path):
    df = pd.read_csv(path, header=0)         # ID, vec_0 … vec_767
    ids = df.iloc[:, 0].astype(str)
    vec = df.iloc[:, 1:].astype(np.float32).to_numpy()
    vec /= np.linalg.norm(vec, axis=1, keepdims=True)   # L2 정규화 보장
    return {k: v for k, v in zip(ids, vec)}

dd_map = load_embed_csv(CFG["DD_EMB_CSV"])
fx_map = load_embed_csv(CFG["FX_EMB_CSV"])
print(f"✅ DD({len(dd_map)}) / Flux({len(fx_map)}) 임베딩 로드 완료")

test_df = pd.read_csv(CFG["CSV_TEST"])
assert {"ID","caption","input_img_path"}.issubset(test_df.columns)

# ============================================================
# 5) 출력 폴더 준비
# ============================================================
os.makedirs(CFG["DIR_OUT_IMG"], exist_ok=True)

# ============================================================
# 6) 메인 루프 – 배치 단위로 clip score 비교
# ============================================================
chosen_rows = []                   # [image_id, 768 floats]

batch_caps, batch_ids = [], []

def flush():
    if not batch_ids: return
    cap_emb = encode_text(batch_caps)
    dd_emb  = torch.tensor(np.stack([dd_map[i] for i in batch_ids]), device=device)
    fx_emb  = torch.tensor(np.stack([fx_map[i] for i in batch_ids]), device=device)

    score_dd = (dd_emb * cap_emb).sum(-1)
    score_fx = (fx_emb * cap_emb).sum(-1)

    for idx, img_id in enumerate(batch_ids):
        use_flux = score_fx[idx] >= score_dd[idx]

        src_img = os.path.join(CFG["DIR_FX_IMG" if use_flux else "DIR_DD_IMG"], img_id + CFG["EXT"])
        dst_img = os.path.join(CFG["DIR_OUT_IMG"],                                           img_id + CFG["EXT"])
        shutil.copyfile(src_img, dst_img)

        emb_vec = fx_emb[idx] if use_flux else dd_emb[idx]
        chosen_rows.append([img_id, *emb_vec.cpu().numpy()])

    batch_caps.clear(); batch_ids.clear()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="🔍 selecting"):
    img_id = os.path.splitext(os.path.basename(row["input_img_path"]))[0]
    if img_id not in dd_map or img_id not in fx_map:
        print(f"⚠️ {img_id} 누락 → skip"); continue
    batch_caps.append(row["caption"])
    batch_ids.append(img_id)
    if len(batch_ids) == CFG["BATCH"]:
        flush()
flush()

# ============================================================
# 7) embed_best.csv 저장
# ============================================================
cols = ["ID"] + [f"vec_{i}" for i in range(768)]
pd.DataFrame(chosen_rows, columns=cols).to_csv(
    CSV_OUT,               # → "/content/drive/.../embed_best.csv"
    index=False            # 행 번호 미포함
    # header=True 가 기본값 → 컬럼명이 첫 줄로 저장
)

print(f"✅ 선택 완료: {len(chosen_rows)}개 이미지·임베딩 저장")


Mounted at /content/drive
💻 device: cuda


/usr/local/lib/python3.11/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


✅ DD(200) / Flux(200) 임베딩 로드 완료


🔍 selecting:   0%|          | 0/200 [00:00<?, ?it/s]

✅ 선택 완료: 200개 이미지·임베딩 저장


In [19]:
import os, zipfile

DIR_OUT_IMG = "/content/drive/MyDrive/INHA_AI_Challenge/open/best_image"
CSV_OUT     = "/content/drive/MyDrive/INHA_AI_Challenge/open/embed_best.csv"   # 실제 위치
zip_path    = "/content/drive/MyDrive/INHA_AI_Challenge/open/submission1.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    # 이미지 200장
    for fname in os.listdir(DIR_OUT_IMG):
        fpath = os.path.join(DIR_OUT_IMG, fname)
        if os.path.isfile(fpath) and not fname.startswith("."):
            z.write(fpath, arcname=fname)               # 폴더 없이
    # 임베딩 CSV (ZIP 내부 이름만 변경)
    z.write(CSV_OUT, arcname="embed_submission.csv")

print(f"✅ 압축 완료: {zip_path}")


✅ 압축 완료: /content/drive/MyDrive/INHA_AI_Challenge/open/submission1.zip
